In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()

# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill

In [2]:
llm= EasyLLM()
agent=BasicAgent(name="test_skill", llm=llm,verbose_thinking=True)
agent.with_skill(CalculatorSkill())
print(llm.model)

gemini-3-flash


In [3]:
#自定义skill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""
agent.with_skill(TranslateSkill())


In [ ]:

from core import enable_logging
enable_logging()
agent.clear_history()
print(agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22" ))

“你是谁，在哪里”翻译为英语是：“Who are you and where are you?”。
这个翻译是正确的，它准确地传达了原句的询问对象和地点的意思。

3^22 的计算结果是：31,381,059,609。


In [4]:


agent.clear_history()
print(agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里"))

INFO:core.agent:对话历史已清空
INFO:agent.BasicAgent:使用工具模式调用智能体
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 400 Bad Request"
ERROR:core.providers.base:❌ openai Provider 工具调用失败: Error code: 400 - {'error': {'message': "Model 'gpt-5.2' is not supported.", 'type': 'upstream_error', 'param': '', 'code': 'UNSUPPORTED_MODEL'}}
ERROR:core.llm:LLM工具调用失败 当前消息{'role': 'user', 'content': '使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里'}
ERROR:agent.BasicAgent:智能体调用失败: Error code: 400 - {'error': {'message': "Model 'gpt-5.2' is not supported.", 'type': 'upstream_error', 'param': '', 'code': 'UNSUPPORTED_MODEL'}}


智能体调用失败: Error code: 400 - {'error': {'message': "Model 'gpt-5.2' is not supported.", 'type': 'upstream_error', 'param': '', 'code': 'UNSUPPORTED_MODEL'}}


In [4]:
manager=agent.skill_manager
prompt=manager.build_skills_prompt()
print(prompt)

你拥有以下技能：
<skills>

## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言

## 数学计算能力
你具备精确的数学计算能力。当遇到以下情况时，请使用计算器工具：
- 复杂的数学运算（大数乘除、幂运算、开方等）
- 统计计算（平均值、标准差等）
- 单位换算或比例计算
- 任何需要精确数值结果的场景
注意：简单的加减运算可以直接心算回答，无需调用工具。


In [ ]:
from skill.registry import SkillRegistry
skill_manage=SkillRegistry()
registered_names = skill_manage.discover_from_directory("./test_skills/")


In [ ]:
print(skill_manage.list_available())

In [ ]:
from skill.folder_loader import FolderSkillLoader
c_skill=FolderSkillLoader.load("./real_skills/crypto_skill/")

In [ ]:
print(c_skill.get_prompt())

In [ ]:
skill_manage.discover_from_directory("./real_skills/")

In [ ]:
print(skill_manage.list_available())


In [ ]:
crypto_skill=skill_manage.create('crypto_skill')
agent.with_skill(crypto_skill)
print(agent.get_enhanced_prompt())

In [ ]:
agent.invoke("i am a boy from china的 SHA-256 哈希值是什么")

In [ ]:
from memory.V2.WorkingMemory import WorkingMemory
from memory import MemoryConfig,MemoryManage
from memory.V2.Embedding.HuggingfaceEmbeddingModel import HuggingfaceEmbeddingModel
config = MemoryConfig(max_capacity=20)
working_memory = WorkingMemory(config)
mm = MemoryManage(
            config=config,
            user_id="test_integration_user",
            enable_working=True,
            working_memory=working_memory,
            enable_episodic=False,
            enable_semantic=False,
            enable_perceptual=False,
        ) 

In [ ]:
agent.with_memory(mm)
print(agent.get_enhanced_prompt())

In [5]:
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill

# 1. 把所有 Skill 注册到全局 Registry（启动时一次性完成）
registry = SkillRegistry.instance()
registry.register_class(CalculatorSkill)
# 为搜索提供元信息
registry.update_metadata("calculator", description="数学计算工具", tags=["math", "compute"])
registry.discover_from_directory("./real_skills/")
# 也可以从目录批量发现
# registry.discover_from_directory("./skills/")

# 2. 创建 Agent（不预加载任何 Skill）
agent1 = BasicAgent(name="assistant", llm=llm, verbose_thinking=True)
agent1.with_skill(MetaSkill(registry,manager=agent1.skill_manager))
print(agent1.get_enhanced_prompt())

INFO:skill.registry:从目录 './real_skills/' 发现并注册 1 个 Skill: ['crypto_skill']
INFO:agent.BasicAgent:BasicAgent 'assistant' 初始化完成，工具调用: 禁用，异步执行: 禁用，provider: openai_responses
INFO:skill.manager:📦 注册 Skill 'meta_skill' (v1.0.0)
INFO:skill.manager:✅ 激活 Skill 'meta_skill' (工具: ['skill_discovery_tool', 'load_skill_tool', 'unload_skill_tool'])


你是一个智能助手，具备使用工具解决问题的能力。

            ## 核心原则
            1. **先思考，再行动**：在调用工具前，先分析用户需求，确定是否需要使用工具
            2. **选择合适的工具**：根据任务需求选择最适合的工具
            3. **正确传递参数**：确保传递给工具的参数格式正确、内容准确
            4. **处理工具结果**：根据工具返回的结果并分析，继续推理或给出最终答案
            5. 在申请工具调用或者回复的同时，需要给出思考过程
            ## 工具使用指南
            - 当用户问题可以直接回答时，不必使用工具
            - 当需要获取实时信息、执行计算或操作外部系统时，使用工具
            - 可以连续调用多个工具来完成复杂任务
            - 如果工具调用失败，分析原因并尝试其他方案
            - 当收集到足够的信息后回答用户问题

            ## 可用工具
            [{'type': 'tool', 'name': 'skill_discovery_tool', 'description': '获取所有可用的额外技能包(Skill)列表及描述。当你发现当前工具箱中没有合适的工具时，调用此工具获取可加载的技能列表。', 'parameters': {'properties': {}, 'title': 'SkillDiscoveryParams', 'type': 'object'}}, {'type': 'tool', 'name': 'load_skill_tool', 'description': '加载一个技能包(Skill)到当前工具箱。请先通过 skill_discovery_tool 获取可用 Skill，然后使用返回的 name 调用此工具加载。', 'parameters': {'properties': {'skill_name': {'description': '要加载的 Skill 注册名称（从 skill_discovery_tool 的返回结果中获取）', 'title': 'Skill Name', 'ty

In [7]:
agent1.invoke("i am a boy from china的 SHA-256 哈希值是什么")

INFO:agent.BasicAgent:使用工具模式调用智能体
INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
INFO:core.providers.openai_responses_provider:📦 Responses API output 类型列表: ['reasoning', 'message', 'function_call']
INFO:core.providers.openai_responses_provider:✅ openairesponses Provider 工具调用响应成功
INFO:agent.BasicAgent:📦 response.output 类型列表: ['reasoning', 'message', 'function_call']
INFO:agent.BasicAgent:💭 模型思考: **Figuring out hashing**

I’m looking for an answer to a hash question. Since there's no direct tool for computing hashes, I might think about doing it manually. Maybe there's a hash calculator I could load as a skill. I need to consider my thought process here since the developer says to provide reasoning for using tools, but I can’t reveal too much about my internal process. I’ll explore options to discover if any relevant skills exist.
INFO:agent.BasicAgent:assistant执行工具: skill_discovery_tool，参数: {}


formatted_response:[ResponseReasoningItem(id='rs_00702b43b3f3cd550169d399b829dc8191bebcefa068198765', summary=[Summary(text="**Figuring out hashing**\n\nI’m looking for an answer to a hash question. Since there's no direct tool for computing hashes, I might think about doing it manually. Maybe there's a hash calculator I could load as a skill. I need to consider my thought process here since the developer says to provide reasoning for using tools, but I can’t reveal too much about my internal process. I’ll explore options to discover if any relevant skills exist.", type='summary_text')], type='reasoning', content=None, encrypted_content='gAAAAABp05m9GH8GgD6pjVHjksuJKZKAZ-Bx12DGM2FTC4lLmxRHZ-WBlqGd7pwuPb9kut8zk3WSwW_0M9SlYSkzxEP91pSuyxzSuo4Bw6P_YZhcXc7a6xnRUDzqAufpklH8ffbntDDi7lwLeeYHHP8MR0THM9Ft4kwMmswjUIr0BIV-js4MbCn964LDId99IhmvbHClOVfoE2s5BeiVyUOkeKLiDANHLA66nZe7DUtJbv_GYZF72DjBQL_EPZ41GA6eanCrE2E_wfOcXzKBzl6zm772SSeqfqlQdUtJIfCoCbgX8UAKlkNNgGtvEk6LeXNYVOjBJuVVYckLvo7B6Yg-he6zAVtL8B

INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
INFO:core.providers.openai_responses_provider:📦 Responses API output 类型列表: ['reasoning', 'message', 'function_call']
INFO:core.providers.openai_responses_provider:✅ openairesponses Provider 工具调用响应成功
INFO:agent.BasicAgent:📦 response.output 类型列表: ['reasoning', 'message', 'function_call']
INFO:agent.BasicAgent:💭 模型思考: 已找到合适的技能包：`crypto_skill`，它支持密码学和哈希计算。我现在加载它来计算该字符串的 SHA-256。
INFO:agent.BasicAgent:assistant执行工具: load_skill_tool，参数: {'skill_name': 'crypto_skill'}
INFO:skill.manager:📦 注册 Skill 'crypto_skill' (v1.0.0)
INFO:skill.manager:✅ 激活 Skill 'crypto_skill' (工具: ['hash_calculator'])


formatted_response:[ResponseReasoningItem(id='rs_00702b43b3f3cd550169d399bebea081919a85b15a99c88560', summary=[], type='reasoning', content=None, encrypted_content='gAAAAABp05nAVPQ42PRlH4zemI33Yf6rLNxpwg3EKH0Jg61MZ6bP4Dl3RA9dqcMTn8uV8hrpH18ZpQZJjUINFNXnDaqtW7yfcX04ai4n0xrBAGnMYbIOONNxp_ejh4R-6gelY-FluWfdB50GB2kxL4_mjxKKDlzMW6W9DRCnX4R4nR82fWysFIiwwehAMTbuRWAZGUNCb934OFK5_t_B88MA3CJNis7Dqj9uA5ovId6r2ej9A1ZkOga0CxXbW2mgD2UZM_-pEICAHrVKCyaiOQbYM3rYlneIpitpZGhlvKgbx6bRnxj3K9_pJEnGzHcICY_afZKzzQ57KRi2ClobnH7clrZwSxKlZvhddpP9wMQotG_q5iATUb05ckX2KQ1K3Y1m3CgdqSaNTpvHV-jwUIpSajcJFnTnnD5fJgAmpxlt804YGw6OOquWkJb1Kl_5w9dAvtJ9ERok-TZuSW-DQtajEfMuj6WmleniRp1SOUZE6RY4Fct1a4WSaRtVtRJlavOAWD-dOTuaG7OiiGMk2sy8teFpCWfUUQrQp0aFj5pU02krrJC63rO-rch1NVRTWpJhrbLmFU5mrHl6hLEC4uKBwc7v3sUQE78MyEN5x0QrDNFixOwmXqt5omrghzN9V4ZLtZh2vy9RuPsRPl89mutpxNwoN5JINT25L-Ne1inrRcBzkjs2M_n4WizRV_uO_jTFWcn34SHikX6mEBE-hbCedTllN1wVg3Am1K83j02YqzEH8ycD2vMS8SsKFQmKDjbabeAT6aENT329z61oYmfC48JRJtZz48uytmY5EM8-gYnEA6YKwjeqhm--z_6A_SQ

INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
INFO:core.providers.openai_responses_provider:📦 Responses API output 类型列表: ['reasoning', 'message', 'function_call']
INFO:core.providers.openai_responses_provider:✅ openairesponses Provider 工具调用响应成功
INFO:agent.BasicAgent:📦 response.output 类型列表: ['reasoning', 'message', 'function_call']
INFO:agent.BasicAgent:💭 模型思考: 我将按你提供的原字符串原样计算，即：

`i am a boy from china`

注意大小写和空格都会影响结果。
INFO:agent.BasicAgent:assistant执行工具: hash_calculator，参数: {'text': 'i am a boy from china'}


...

INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
INFO:core.providers.openai_responses_provider:📦 Responses API output 类型列表: ['reasoning', 'message', 'function_call']
INFO:core.providers.openai_responses_provider:✅ openairesponses Provider 工具调用响应成功
INFO:agent.BasicAgent:📦 response.output 类型列表: ['reasoning', 'message', 'function_call']
INFO:agent.BasicAgent:💭 模型思考: 哈希值已经算出。为了释放资源，我卸载刚才加载的技能包。
INFO:agent.BasicAgent:assistant执行工具: unload_skill_tool，参数: {'skill_name': 'crypto_skill'}
INFO:skill.manager:⏸️  停用 Skill 'crypto_skill'
INFO:skill.manager:📦 注销 Skill 'crypto_skill'


formatted_response:[ResponseReasoningItem(id='rs_00702b43b3f3cd550169d399c60b28819189a240fd210524c8', summary=[], type='reasoning', content=None, encrypted_content='gAAAAABp05nIOLP6cEw4Z0iBlbkW50A2NftqZarsEZ-Qw79BV-DCP1fOo3AoKHGuxEoB6L61ELtx8B9YNdJx-Aoz_ZyQ8-hIE3qeTzDQDYW7oR7pzEc95z7kDn4Uldld7tbECLmF7rnsju_KLPomrLPgVZIeD0Y9FX9LNqk3XyvpxYLQk1Z8CAgOii-SVG2PA-sj72kOyzXgVCKQkAs7h64jfU90ODJxA-GI7_XhCk3uM99u9b4-ZpLVh3T4sd3VnFyCDm46uVLiE4t-48XoowHYyD9T5LVlcJxsbVkXKoFJCxT8AsqnAuift0ZZmaSNFXM5USNXHV1qswEV40I_b3alYeIqdqPT0imNisFxJyCkvDt7zaR0Kf_lBAQvt2OaxKeixm_Jkl_QUK-Kryt9lfSx9ZKARN65DMESbpHFMTtD-hFEKWCapWCS9W0xjx5UX05eRKljXPm9TaMpiEjNJM2__sdRYfZAuN7YtGFunFpJFz9z5yvwOaXVHO5oCa0G7TEQDUjupL7mTK13gKlUdVSe6gdwSM0dzEyTsXL9ttg7NSJlTaoHfscqZCSlORbN4Evx6HHCq-1igTF-9ErwO-0HZQM0Xu1AHH5QdqbcuJiJzYT94zApNRrSkbJip7rTW9yuRUJrF_IWwT-u3arq1Zce2Tp4XjJKodvu7SEIw5jZDFuJlAtYjMDxn4cnarD6Kv5gDuJPxTGRsntJxLDVZc7w7wHp5qAsOm3iksC9N1BdC-xH5qtQNsz7bBC4OVJAZTe0NtZi4HF1KN26Y4tOQKzmTI8n49ZP5a-1WQi3Tv12SiqOM1Zsf0tl4NXFNfw89wH

INFO:httpx:HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
INFO:core.providers.openai_responses_provider:📦 Responses API output 类型列表: ['message']
INFO:core.providers.openai_responses_provider:✅ openairesponses Provider 工具调用响应成功
INFO:agent.BasicAgent:📦 response.output 类型列表: ['message']


formatted_response:[ResponseOutputMessage(id='msg_00702b43b3f3cd550169d399ca77588191b79242dbbe9042ca', content=[ResponseOutputText(annotations=[], text='`i am a boy from china` 的 SHA-256 哈希值是：\n\n**3ae42c9aabe1ab4bfdd627e79447b1a388994a9dfc32c2f9ce656d7ff156b32d**\n\n如果你愿意，我也可以顺便帮你算：\n- 加上引号后的哈希\n- 去掉空格后的哈希\n- 中文字符串的 SHA-256', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]
get_thinking_content:[ResponseOutputMessage(id='msg_00702b43b3f3cd550169d399ca77588191b79242dbbe9042ca', content=[ResponseOutputText(annotations=[], text='`i am a boy from china` 的 SHA-256 哈希值是：\n\n**3ae42c9aabe1ab4bfdd627e79447b1a388994a9dfc32c2f9ce656d7ff156b32d**\n\n如果你愿意，我也可以顺便帮你算：\n- 加上引号后的哈希\n- 去掉空格后的哈希\n- 中文字符串的 SHA-256', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


'`i am a boy from china` 的 SHA-256 哈希值是：\n\n**3ae42c9aabe1ab4bfdd627e79447b1a388994a9dfc32c2f9ce656d7ff156b32d**\n\n如果你愿意，我也可以顺便帮你算：\n- 加上引号后的哈希\n- 去掉空格后的哈希\n- 中文字符串的 SHA-256'